# 🎯 10. GraphRAG Architectures (Deep Dive)

Στο προηγούμενο notebook είδαμε πώς να κάνουμε extract ένα γράφημα. 
Αυτό όμως δεν είναι RAG. RAG σημαίνει **Ανάκτηση (Retrieval)**.

Σε αυτό το notebook θα γράψουμε πραγματικό κώδικα που:
1. Δημιουργεί ένα in-memory `networkx` γράφημα.
2. Δέχεται ένα user query και εξάγει την κύρια οντότητα.
3. Κάνει **2-hop Graph Traversal** για να μαζέψει όλο το context (κόμβους και σχέσεις) γύρω από αυτή την οντότητα.
4. Ταΐζει αυτό το context στο LLM για να απαντήσει!

## 10.1 Setup & Δημιουργία του Knowledge Graph

### Ένας γράφος ενος ΥΠΟΘΕΤΙΚΟΥ ΣΕΝΑΡΙΟΥ

<img src="images/rel-graph.png" width="70%" style="border-radius:10px;margin:12px 0;"/>


In [2]:
import networkx as nx
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

load_dotenv()

# Ας υποθέσουμε ότι το GraphRAG Extraction step (από το προηγούμενο notebook) 
# μας έδωσε τα εξής δεδομένα:
relations = [
    ("Sam Altman", "CEO", "OpenAI"),
    ("OpenAI", "δημιούργησε", "GPT-4"),
    ("Microsoft", "επένδυσε", "OpenAI"),
    ("Satya Nadella", "CEO", "Microsoft"),
    ("Satya Nadella", "συνεργάζεται", "Sam Altman"),
    ("GPT-4", "χρησιμοποιείται", "Copilot")
]

# Χτίζουμε το γράφημα
G = nx.DiGraph()
for source, rel, target in relations:
    G.add_edge(source, target, relation=rel)

print(f"Το γράφημα δημιουργήθηκε με {G.number_of_nodes()} κόμβους και {G.number_of_edges()} ακμές.")

Το γράφημα δημιουργήθηκε με 6 κόμβους και 6 ακμές.


## 10.2 Graph Traversal Retriever
Όταν ο χρήστης κάνει μια ερώτηση (π.χ. "Πώς σχετίζεται ο Satya Nadella με το GPT-4;"), χρειαζόμαστε 2 βήματα:
1. Να βρούμε το **Entity** (π.χ. "Satya Nadella") από την ερώτηση.
2. Να περπατήσουμε (traverse) το γράφημα για να βρούμε συνδέσεις βάθους 1 ή 2 (n-hops).

In [3]:
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ΒΗΜΑ 1: Entity Extraction από το Query
class QueryEntity(BaseModel):
    entity: str = Field(description="Το κύριο πρόσωπο, εταιρεία ή αντικείμενο στην ερώτηση.")

extract_prompt = ChatPromptTemplate.from_template("Εξήγαγε την κύρια οντότητα από την ερώτηση: {query}")
entity_extractor = extract_prompt | llm.with_structured_output(QueryEntity)

# ΒΗΜΑ 2: 2-Hop Traversal (BFS algorithm)
def traverse_graph(graph, start_node, max_depth=2):
    if start_node not in graph:
        return "Οντότητα δεν βρέθηκε στο Knowledge Graph."
    
    visited = set()
    queue = [(start_node, 0)]
    context_paths = []
    
    while queue:
        current_node, depth = queue.pop(0)
        if depth >= max_depth:
            continue
            
        if current_node not in visited:
            visited.add(current_node)
            
            # Βρες τους γείτονες
            for neighbor in graph.neighbors(current_node):
                rel = graph[current_node][neighbor]['relation']
                path_str = f"{current_node} --[{rel}]--> {neighbor}"
                if path_str not in context_paths:
                    context_paths.append(path_str)
                queue.append((neighbor, depth + 1))
                
            # Βρες και όσους δείχνουν ΠΡΟΣ αυτόν τον κόμβο (in-edges)
            for predecessor in graph.predecessors(current_node):
                rel = graph[predecessor][current_node]['relation']
                path_str = f"{predecessor} --[{rel}]--> {current_node}"
                if path_str not in context_paths:
                    context_paths.append(path_str)
                queue.append((predecessor, depth + 1))
                
    return "\n".join(context_paths)

print("Graph Retriever function ready!")

Graph Retriever function ready!


## 10.3 Graph Generation (QA)
Τώρα που έχουμε το context από το Graph, το δίνουμε στο LLM για να απαντήσει!

In [4]:
qa_prompt = ChatPromptTemplate.from_template("""
Απάντησε στην ερώτηση χρησιμοποιώντας ΜΟΝΟ το παρακάτω Knowledge Graph Context.

Context:
{context}

Ερώτηση: {query}
""")

qa_chain = qa_prompt | llm

def graph_rag_pipeline(query: str):
    print(f"\nΕρώτηση: {query}")
    
    # 1. Βρες την Οντότητα
    extracted = entity_extractor.invoke({"query": query})
    start_entity = extracted.entity
    print(f"[Router] Εξήχθη Οντότητα: {start_entity}")
    
    # 2. Κάνε Traversal (Retrieval)
    context = traverse_graph(G, start_entity, max_depth=2)
    print(f"[Retriever] Βρέθηκε το εξής Graph Context:\n{context}")
    
    if "δεν βρέθηκε" in context:
        return "Συγγνώμη, δεν έχω πληροφορίες για αυτό."
    
    # 3. Generate
    answer = qa_chain.invoke({"context": context, "query": query})
    print(f"\n[Generator] Απάντηση:\n{answer.content}")
    return answer.content

# Δοκιμές!
graph_rag_pipeline("Πώς σχετίζεται ο Satya Nadella με το GPT-4;")
print("-"*50)
graph_rag_pipeline("Ποιος δημιούργησε το Copilot;")


Ερώτηση: Πώς σχετίζεται ο Satya Nadella με το GPT-4;
[Router] Εξήχθη Οντότητα: Satya Nadella
[Retriever] Βρέθηκε το εξής Graph Context:
Satya Nadella --[CEO]--> Microsoft
Satya Nadella --[συνεργάζεται]--> Sam Altman
Microsoft --[επένδυσε]--> OpenAI
Sam Altman --[CEO]--> OpenAI

[Generator] Απάντηση:
Ο Satya Nadella είναι ο CEO της Microsoft, η οποία έχει επενδύσει στην OpenAI, της οποίας ο Sam Altman είναι ο CEO. Το GPT-4 αναπτύσσεται από την OpenAI, άρα ο Satya Nadella σχετίζεται με το GPT-4 μέσω της επένδυσης της Microsoft στην OpenAI.
--------------------------------------------------

Ερώτηση: Ποιος δημιούργησε το Copilot;
[Router] Εξήχθη Οντότητα: Copilot
[Retriever] Βρέθηκε το εξής Graph Context:
GPT-4 --[χρησιμοποιείται]--> Copilot
OpenAI --[δημιούργησε]--> GPT-4

[Generator] Απάντηση:
Δεν υπάρχει πληροφορία στο παρακάτω Knowledge Graph Context σχετικά με το ποιος δημιούργησε το Copilot.


'Δεν υπάρχει πληροφορία στο παρακάτω Knowledge Graph Context σχετικά με το ποιος δημιούργησε το Copilot.'

## 📋 Συμπεράσματα
Αυτό το notebook δείχνει την πραγματική καρδιά του GraphRAG:
1. Αντί να κάνουμε embed ολόκληρες παραγράφους, κάνουμε embed τον **Γράφο (Nodes & Edges)**.
2. Ο αλγόριθμος **Traversal (π.χ. BFS 2-hops)** ταξιδεύει από κόμβο σε κόμβο μαζεύοντας πληροφορίες που ίσως βρίσκονταν σε εντελώς διαφορετικά έγγραφα.
3. Το context που φτάνει στο LLM είναι δομημένο και απόλυτα καθαρό, λύνοντας το πρόβλημα του multi-hop reasoning.